# SNI source-group-balanced split v3

Notebook ini **tidak melakukan training** dan tidak membuka test. Ia hanya membuat ulang manifest 70/15/15 agar validation/test seimbang berdasarkan foto sumber, bukan jumlah crop.

In [ ]:
# 1/3 — Setup repository dan Google Drive
from google.colab import drive
from pathlib import Path
import json, subprocess, sys

drive.mount('/content/drive')
REPO = Path('/content/coffee-bean-classification')
BRANCH = 'agent/sni-instance-crops'
if not (REPO / '.git').is_dir():
    subprocess.run([
        'git', 'clone', '--branch', BRANCH, '--single-branch',
        'https://github.com/ediprin/coffee-bean-classification.git', str(REPO)
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)

DRIVE_V1 = Path('/content/drive/MyDrive/coffee-sni-instance-crop-v1')
LOCAL_V1 = Path('/content/sni-instance-crops')
V3_ROOT = DRIVE_V1 / 'classification-v3-source-balanced'
print('REPO  :', REPO)
print('OUTPUT:', V3_ROOT)

In [ ]:
# 2/3 — Cari metadata audit v1 dan bangun split (CPU; tanpa membaca gambar)
candidates = [LOCAL_V1, DRIVE_V1]
INPUT_ROOT = next(
    (root for root in candidates
     if (root / 'audit.json').is_file() and (root / 'manifest.csv').is_file()),
    None,
)
assert INPUT_ROOT is not None, (
    'audit.json dan manifest.csv v1 tidak ditemukan. '
    'Pastikan folder coffee-sni-instance-crop-v1 sudah dibagikan ke akun ini.'
)
print('INPUT:', INPUT_ROOT)

subprocess.run([
    sys.executable, '-u', '-m',
    'bilinear_lmmd.data.preparation.prepare_sni_classification_v3',
    '--input-root', str(INPUT_ROOT),
    '--output-root', str(V3_ROOT),
    '--seed', '42',
    '--trials', '512',
    '--min-eval-samples-per-class', '50',
    '--min-eval-groups-per-class', '20',
    '--min-eval-groups-per-dataset', '50',
    '--max-single-group-fraction-per-class', '0.25',
    '--metadata-only',
], check=True, cwd=REPO)

In [ ]:
# 3/3 — Tampilkan audit; kirim seluruh output ini
audit = json.loads((V3_ROOT / 'audit.json').read_text())
print('\n=== PUTUSAN SPLIT V3 ===')
print('Gate       :', audit['statistical_readiness']['split_gate'])
print('Test locked:', audit['test_locked'])
print('Training   :', audit['training_authorized'])
for split in ('train', 'val', 'test'):
    row = audit['split_statistics'][split]
    print(f"\n{split.upper()}: crops={row['crops']:,} groups={row['source_groups']:,}")
    print('  domain groups:', row['dataset_group_counts'])
    print('  largest group fraction:', f"{row['largest_group_fraction']:.2%}")
print('\nWeak validation classes:')
print(json.dumps(audit['statistical_readiness']['weak_classes']['val'], indent=2))
print('\nWeak test classes:')
print(json.dumps(audit['statistical_readiness']['weak_classes']['test'], indent=2))
print('\nConcentration failures:')
print(json.dumps(audit['statistical_readiness']['concentration_failures'], indent=2))
print('\nSAVED:', V3_ROOT / 'audit.json')
print('\nJangan training. Kirim output audit ini terlebih dahulu.')